# 🚀 Humara Aggressive Humanizer — Step-by-Step Training Guide

Follow each step **in order**. Just click each code cell and press **Shift+Enter** (or the Play button) to run it.

---

## Before You Start

1. Go to [Google Colab](https://colab.research.google.com/)
2. Click **File → Upload Notebook** and upload this `.ipynb` file
3. Click **Runtime → Change runtime type** and select **T4 GPU** (recommended, not required)
4. Run each cell below **one by one** from top to bottom

---

### What This Does

| Step | What Happens |
|------|-------------|
| Steps 1-2 | Install libraries + clone your project |
| Step 3 | Download 200K+ real human vs AI texts from HuggingFace |
| Steps 4-5 | Analyze what makes AI text detectable (15 features) |
| Steps 6-7 | Train an aggressive AI detector on all 200K+ texts |
| Step 8 | Extract top 50 AI patterns + top 50 human patterns |
| Steps 9-10 | Test your humanizer against the detector |
| Step 11 | Run the built-in trainer loop |
| Step 12 | Save 200 new training pairs for future use |
| Step 13 | Test on YOUR own text |
| Step 14 | Download everything as a ZIP |

**Total time: ~15-30 minutes** (depending on internet speed)

---
## STEP 1: Install All Dependencies

This installs Python libraries needed for training. Takes about 2 minutes.

**Just run the cell below** — wait until you see `All dependencies installed!`

In [ ]:
# ============================================
# STEP 1: INSTALL DEPENDENCIES
# Just run this cell. Wait for "All dependencies installed!"
# ============================================

!pip install -q spacy textstat nltk scikit-learn numpy pandas datasets matplotlib seaborn
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)

print()
print('=' * 50)
print('  All dependencies installed!')
print('=' * 50)

---
## STEP 2: Clone Your Project from GitHub

This downloads your humanizer engine code into Colab.

**Just run the cell below** — wait until you see the working directory and file count.

In [ ]:
# ============================================
# STEP 2: CLONE PROJECT
# This downloads your humanizer code from GitHub
# ============================================

import os, sys

if not os.path.exists('/content/repo'):
    !git clone https://github.com/chepdor2-ai/humara-s-clean-canvas.git /content/repo
    print('\nCloned successfully!')
else:
    print('Repo already cloned.')

ENGINE_DIR = '/content/repo/humanizer-engine'
PYBACKUP = os.path.join(ENGINE_DIR, 'python-backup')
os.chdir(PYBACKUP)
sys.path.insert(0, PYBACKUP)

py_files = [f for f in os.listdir('.') if f.endswith('.py')]
print(f'\nWorking directory: {os.getcwd()}')
print(f'Python files found: {len(py_files)}')
print(f'Key files: {[f for f in py_files if f in ["humanizer.py","rules.py","trainer.py","evaluator.py"]]}')

---
## STEP 3: Download Real Human vs AI Datasets (200K+ Texts)

This downloads **4 real datasets** from HuggingFace:

| Dataset | Size | What It Contains |
|---------|------|------------------|
| HC3 | 24K+ pairs | Real human answers vs ChatGPT answers |
| GPT-Wiki-Intro | 20K sample | Real Wikipedia intros vs GPT-generated intros |
| AI-Human-v1 | 52K+ | Mixed AI and human labeled text |
| dmitva | 1.4K | Additional AI vs human text |

**This takes 3-5 minutes.** Wait for the `TOTAL COMBINED DATASET` summary at the bottom.

In [ ]:
# ============================================
# STEP 3: DOWNLOAD ALL DATASETS
# Takes 3-5 min. Wait for the TOTAL at the end.
# ============================================

from datasets import load_dataset
import pandas as pd
import numpy as np

CACHE = '/content/.cache/hf'
os.makedirs('/content/data', exist_ok=True)

all_texts = []
all_labels = []   # 0 = human,  1 = AI
all_sources = []

# --- Dataset 1: HC3 (Human vs ChatGPT) ---
print('\n' + '='*60)
print('[1/4] LOADING: HC3 (Human vs ChatGPT answers)')
print('='*60)
try:
    hc3 = load_dataset('Hello-SimpleAI/HC3', 'all', cache_dir=CACHE, trust_remote_code=True)
    hc3_df = hc3['train'].to_pandas()
    hc3_count = 0
    for _, row in hc3_df.iterrows():
        if isinstance(row.get('human_answers'), list):
            for ans in row['human_answers']:
                if isinstance(ans, str) and len(ans) > 50:
                    all_texts.append(ans[:2000])
                    all_labels.append(0)
                    all_sources.append('hc3_human')
                    hc3_count += 1
        if isinstance(row.get('chatgpt_answers'), list):
            for ans in row['chatgpt_answers']:
                if isinstance(ans, str) and len(ans) > 50:
                    all_texts.append(ans[:2000])
                    all_labels.append(1)
                    all_sources.append('hc3_chatgpt')
                    hc3_count += 1
    print(f'  -> Loaded {hc3_count} samples')
except Exception as e:
    print(f'  -> Failed: {e}')

# --- Dataset 2: GPT-Wiki-Intro ---
print('\n' + '='*60)
print('[2/4] LOADING: GPT-Wiki-Intro (Wikipedia vs GPT)')
print('='*60)
try:
    wiki_gpt = load_dataset('aadityaubhat/GPT-wiki-intro', cache_dir=CACHE, trust_remote_code=True)
    wiki_df = wiki_gpt['train'].to_pandas()
    wiki_sample = wiki_df.sample(n=min(20000, len(wiki_df)), random_state=42)
    wiki_count = 0
    for _, row in wiki_sample.iterrows():
        wiki_text = row.get('wiki_intro', '')
        if isinstance(wiki_text, str) and len(wiki_text) > 50:
            all_texts.append(wiki_text[:2000])
            all_labels.append(0)
            all_sources.append('wiki_human')
            wiki_count += 1
        gpt_text = row.get('generated_intro', '')
        if isinstance(gpt_text, str) and len(gpt_text) > 50:
            all_texts.append(gpt_text[:2000])
            all_labels.append(1)
            all_sources.append('wiki_gpt')
            wiki_count += 1
    print(f'  -> Loaded {wiki_count} samples')
except Exception as e:
    print(f'  -> Failed: {e}')

# --- Dataset 3: AI-Human-Text-Detection-v1 ---
print('\n' + '='*60)
print('[3/4] LOADING: AI-Human-Text-Detection-v1')
print('='*60)
try:
    ai_human = load_dataset('silentone0725/ai-human-text-detection-v1', cache_dir=CACHE, trust_remote_code=True)
    ah_df = ai_human['train'].to_pandas()
    ah_count = 0
    text_col = 'text' if 'text' in ah_df.columns else ah_df.columns[0]
    label_col = None
    for col in ['generated', 'label', 'is_ai', 'class']:
        if col in ah_df.columns:
            label_col = col
            break
    if label_col:
        for _, row in ah_df.iterrows():
            txt = str(row[text_col])
            if len(txt) > 50:
                lbl = int(row[label_col]) if str(row[label_col]).isdigit() else (1 if 'ai' in str(row[label_col]).lower() else 0)
                all_texts.append(txt[:2000])
                all_labels.append(lbl)
                all_sources.append('aihumanv1')
                ah_count += 1
    print(f'  -> Loaded {ah_count} samples')
except Exception as e:
    print(f'  -> Failed: {e}')

# --- Dataset 4: dmitva ---
print('\n' + '='*60)
print('[4/4] LOADING: dmitva/human_ai_generated_text')
print('='*60)
try:
    dmitva = load_dataset('dmitva/human_ai_generated_text', cache_dir=CACHE, trust_remote_code=True)
    dm_df = dmitva['train'].to_pandas()
    dm_count = 0
    for _, row in dm_df.iterrows():
        txt = str(row.get('text', ''))
        if len(txt) > 50:
            all_texts.append(txt[:2000])
            all_labels.append(int(row.get('generated', 0)))
            all_sources.append('dmitva')
            dm_count += 1
    print(f'  -> Loaded {dm_count} samples')
except Exception as e:
    print(f'  -> Failed: {e}')

# --- Final Summary ---
print('\n' + '='*60)
print('  TOTAL COMBINED DATASET')
print('='*60)
print(f'  Total samples:  {len(all_texts):,}')
print(f'  Human texts:    {sum(1 for l in all_labels if l==0):,}')
print(f'  AI texts:       {sum(1 for l in all_labels if l==1):,}')
print(f'\n  Sources breakdown:')
for src, cnt in pd.Series(all_sources).value_counts().items():
    print(f'    {src}: {cnt:,}')

---
## STEP 4: Analyze — What Makes AI Text Detectable?

This profiles 5,000 human texts and 5,000 AI texts across **15 features** to find the strongest signals that distinguish AI from human writing.

**Look at the output table** — features marked `STRONG` are the biggest giveaways of AI text.

In [ ]:
# ============================================
# STEP 4: ANALYZE AI vs HUMAN PATTERNS
# This finds what makes AI text detectable
# ============================================

import textstat
from nltk.tokenize import sent_tokenize
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

def profile_text(text):
    """Extract 15 linguistic features from a text."""
    sents = sent_tokenize(text)
    words = text.split()
    word_lengths = [len(w) for w in words]
    sent_lengths = [len(s.split()) for s in sents] if sents else [0]
    starters = [s.split()[0].lower() for s in sents if s.split()]

    ai_markers = [
        'furthermore', 'moreover', 'additionally', 'consequently',
        'in conclusion', 'it is important', 'it is crucial',
        'it is worth noting', 'in summary', 'to summarize',
        'on the other hand', 'nevertheless', "in today's",
        'in the modern', 'plays a crucial role', 'it is essential',
        'this essay', 'this paper'
    ]
    text_lower = text.lower()
    marker_count = sum(1 for m in ai_markers if m in text_lower)

    bigrams = [' '.join(words[i:i+2]).lower() for i in range(len(words)-1)]
    bigram_counts = Counter(bigrams)
    repeated_bigrams = sum(1 for c in bigram_counts.values() if c >= 3)

    return {
        'avg_sent_len': np.mean(sent_lengths),
        'sent_len_std': np.std(sent_lengths),
        'avg_word_len': np.mean(word_lengths) if word_lengths else 0,
        'vocab_diversity': len(set(w.lower() for w in words)) / max(len(words), 1),
        'flesch_ease': textstat.flesch_reading_ease(text),
        'starter_diversity': len(set(starters)) / max(len(starters), 1),
        'ai_marker_count': marker_count,
        'ai_marker_density': marker_count / max(len(sents), 1),
        'repeated_bigrams': repeated_bigrams,
        'num_sentences': len(sents),
        'has_questions': 1 if '?' in text else 0,
        'has_contractions': 1 if any(c in text_lower for c in ["it's", "don't", "can't", "won't", "isn't", "we're", "they're"]) else 0,
        'has_first_person': 1 if any(p in text_lower.split() for p in ['i', 'we', 'my', 'our', 'me', 'us']) else 0,
        'dash_count': text.count('\u2014') + text.count('\u2013') + text.count(' - '),
        'paren_count': text.count('('),
    }

# Profile 5000 of each
PROFILE_N = 5000
human_indices = [i for i, l in enumerate(all_labels) if l == 0]
ai_indices = [i for i, l in enumerate(all_labels) if l == 1]

np.random.seed(42)
human_sample_idx = np.random.choice(human_indices, size=min(PROFILE_N, len(human_indices)), replace=False)
ai_sample_idx = np.random.choice(ai_indices, size=min(PROFILE_N, len(ai_indices)), replace=False)

print(f'Profiling {len(human_sample_idx)} human + {len(ai_sample_idx)} AI texts...\n')

human_profiles = [profile_text(all_texts[i]) for i in human_sample_idx]
ai_profiles = [profile_text(all_texts[i]) for i in ai_sample_idx]

features = list(human_profiles[0].keys())
print(f'{"FEATURE":<25} {"HUMAN":>10} {"AI":>10} {"DIFF":>10} {"SIGNAL":>12}')
print('-' * 72)

signals = {}
for feat in features:
    h_vals = [p[feat] for p in human_profiles]
    a_vals = [p[feat] for p in ai_profiles]
    h_avg = np.mean(h_vals)
    a_avg = np.mean(a_vals)
    diff = a_avg - h_avg
    pooled_std = np.sqrt((np.std(h_vals)**2 + np.std(a_vals)**2) / 2) or 1
    cohens_d = abs(diff) / pooled_std
    signals[feat] = cohens_d

    tag = '** STRONG **' if cohens_d > 0.5 else ('  moderate' if cohens_d > 0.2 else '  weak')
    print(f'{feat:<25} {h_avg:>10.3f} {a_avg:>10.3f} {diff:>+10.3f} {tag}')

print('\n** STRONG = biggest AI giveaways.  Your humanizer must fix these. **')

---
## STEP 5: Visualize the Differences

This creates histograms showing how human and AI text differ across the top 8 features.

**Green = Human, Red = AI.** Where the colors overlap = harder to detect.

In [ ]:
# ============================================
# STEP 5: VISUALIZE FEATURE DISTRIBUTIONS
# Green = Human, Red = AI
# ============================================

import matplotlib.pyplot as plt
import seaborn as sns

top_signals = sorted(signals.items(), key=lambda x: x[1], reverse=True)[:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for idx, (feat, d) in enumerate(top_signals):
    h_vals = [p[feat] for p in human_profiles]
    a_vals = [p[feat] for p in ai_profiles]
    axes[idx].hist(h_vals, bins=40, alpha=0.6, label='Human', color='#2ecc71', density=True)
    axes[idx].hist(a_vals, bins=40, alpha=0.6, label='AI', color='#e74c3c', density=True)
    axes[idx].set_title(f'{feat}\n(signal={d:.2f})', fontsize=10)
    axes[idx].legend(fontsize=8)
    axes[idx].grid(True, alpha=0.2)

plt.suptitle('Human vs AI Text Features (Top 8 Signals)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/ai_vs_human_features.png', dpi=150)
plt.show()
print('\nPlot saved to /content/ai_vs_human_features.png')

---
## STEP 6: Train the Aggressive AI Detector (Part 1 — Build Features)

This builds a feature matrix from ALL 200K+ texts using:
- **TF-IDF** (15,000 word/bigram features)
- **15 linguistic features** from Step 4

**Takes 5-10 minutes.** You'll see progress updates.

In [ ]:
# ============================================
# STEP 6: BUILD FEATURE MATRIX
# Takes 5-10 min. Watch the progress.
# ============================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from scipy.sparse import hstack
import pickle

# Split into train/test
X_train_text, X_test_text, y_train, y_test, src_train, src_test = train_test_split(
    all_texts, all_labels, all_sources,
    test_size=0.15, random_state=42, stratify=all_labels
)

print(f'Train set: {len(X_train_text):,} samples')
print(f'Test set:  {len(X_test_text):,} samples')
print(f'AI ratio:  {sum(y_train)/len(y_train)*100:.1f}%')

# TF-IDF features
print('\n[1/3] Fitting TF-IDF (15K features, 1-2 grams)...')
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    sublinear_tf=True,
)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)
print(f'  TF-IDF shape: {X_train_tfidf.shape}')

# Linguistic features
print('\n[2/3] Extracting linguistic features for train set...')
def extract_features_batch(texts, label=''):
    feats = []
    for i, t in enumerate(texts):
        if i % 5000 == 0 and i > 0:
            print(f'  ...{label} {i:,}/{len(texts):,}')
        feats.append(profile_text(t))
    return pd.DataFrame(feats)

train_ling = extract_features_batch(X_train_text, 'train')
print(f'\n[3/3] Extracting linguistic features for test set...')
test_ling = extract_features_batch(X_test_text, 'test')

# Combine
X_train_combined = hstack([X_train_tfidf, train_ling.values])
X_test_combined = hstack([X_test_tfidf, test_ling.values])

print(f'\nFinal feature matrix: {X_train_combined.shape[0]:,} samples x {X_train_combined.shape[1]:,} features')
print('Ready to train!')

---
## STEP 7: Train the Aggressive AI Detector (Part 2 — Train Model)

This trains a **Logistic Regression** model on the combined features.

**Look at the results** — you want AUC close to 1.0 and high precision/recall.

In [ ]:
# ============================================
# STEP 7: TRAIN LOGISTIC REGRESSION DETECTOR
# ============================================

print('Training Logistic Regression on all features...')
lr_model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    C=1.0,
    random_state=42,
    solver='saga',
    n_jobs=-1,
)
lr_model.fit(X_train_combined, y_train)

# Evaluate
lr_preds = lr_model.predict(X_test_combined)
lr_probs = lr_model.predict_proba(X_test_combined)[:, 1]
lr_auc = roc_auc_score(y_test, lr_probs)

print()
print('=' * 60)
print('   DETECTOR RESULTS')
print('=' * 60)
print(f'   AUC Score: {lr_auc:.4f}  (closer to 1.0 = better)')
print()
print(classification_report(y_test, lr_preds, target_names=['Human', 'AI']))

cm = confusion_matrix(y_test, lr_preds)
print(f'Confusion Matrix:')
print(f'  True Negatives  (human correctly identified): {cm[0,0]:>6,}')
print(f'  False Positives (human wrongly flagged as AI): {cm[0,1]:>6,}')
print(f'  False Negatives (AI text missed):              {cm[1,0]:>6,}')
print(f'  True Positives  (AI correctly caught):         {cm[1,1]:>6,}')

# Save model
os.makedirs('/content/models', exist_ok=True)
with open('/content/models/aggressive_detector.pkl', 'wb') as f:
    pickle.dump({
        'model': lr_model,
        'tfidf': tfidf,
        'auc': lr_auc,
        'train_size': len(X_train_text),
        'features': 'tfidf_15k + linguistic_15feat',
    }, f)

print(f'\nModel saved: /content/models/aggressive_detector.pkl')

---
## STEP 8: Extract AI Patterns — What Gets Caught

This mines the trained model to find:
- **Top 50 features** that scream "this is AI"
- **Top 50 features** that scream "this is human"

This is the **key output** — it tells you exactly what your humanizer needs to fix.

In [ ]:
# ============================================
# STEP 8: EXTRACT AI vs HUMAN PATTERNS
# This shows what detectors actually catch
# ============================================

import json

feature_names = tfidf.get_feature_names_out().tolist()
ling_names = list(train_ling.columns)
all_feature_names = feature_names + ling_names

coefficients = lr_model.coef_[0]

ai_top_idx = np.argsort(coefficients)[-50:][::-1]
human_top_idx = np.argsort(coefficients)[:50]

# Print AI indicators
print('=' * 60)
print('  TOP 50 AI INDICATORS (words/features that scream AI)')
print('=' * 60)
ai_patterns = []
for i, idx in enumerate(ai_top_idx):
    name = all_feature_names[idx] if idx < len(all_feature_names) else f'feature_{idx}'
    coef = coefficients[idx]
    ai_patterns.append({'feature': name, 'weight': float(coef)})
    print(f'  {i+1:>2}. {name:<40} +{coef:.4f}')

# Print Human indicators
print(f'\n{"="*60}')
print('  TOP 50 HUMAN INDICATORS (words/features that scream human)')
print('=' * 60)
human_patterns = []
for i, idx in enumerate(human_top_idx):
    name = all_feature_names[idx] if idx < len(all_feature_names) else f'feature_{idx}'
    coef = coefficients[idx]
    human_patterns.append({'feature': name, 'weight': float(coef)})
    print(f'  {i+1:>2}. {name:<40} {coef:.4f}')

# Save report
pattern_report = {
    'ai_indicators': ai_patterns[:30],
    'human_indicators': human_patterns[:30],
    'training_stats': {
        'total_samples': len(all_texts),
        'human_count': sum(1 for l in all_labels if l == 0),
        'ai_count': sum(1 for l in all_labels if l == 1),
        'auc_score': float(lr_auc),
        'datasets': list(set(all_sources)),
    },
    'signal_strengths': {k: round(v, 3) for k, v in sorted(signals.items(), key=lambda x: -x[1])},
}

with open('/content/ai_pattern_report.json', 'w') as f:
    json.dump(pattern_report, f, indent=2)

print(f'\nPattern report saved: /content/ai_pattern_report.json')
print()
print('KEY TAKEAWAY:')
print('  AI = uniform sentences, formal transitions, no questions/contractions')
print('  Human = varied rhythm, questions, dashes, contractions, first person')

---
## STEP 9: Test Humanizer Against the Detector

This takes 200 real AI texts and runs them through your humanizer at 3 strength levels (light / medium / strong), then checks if the detector can still catch them.

- **Bypass rate > 50%** = good  
- **Bypass rate > 80%** = excellent

**If the humanizer can't load** (missing dependencies), this will skip and still show detector results.

In [ ]:
# ============================================
# STEP 9: SHOW CURRENT RULE PARAMETERS
# ============================================

import rules

print('Your current humanizer rule parameters:')
print('-' * 40)
for param in ['BURSTINESS_TARGET', 'SYNONYM_RATE', 'PHRASE_RATE', 'RESTRUCTURE_RATE',
              'CLAUSE_SWAP_RATE', 'CONTRACTION_RATE', 'TRANSITION_RATE', 'SHORTEN_RATE']:
    if hasattr(rules, param):
        print(f'  {param:22s} = {getattr(rules, param)}')
    else:
        print(f'  {param:22s} = (not found)')

In [ ]:
# ============================================
# STEP 9 (continued): TEST HUMANIZER vs DETECTOR
# This tests 50 AI texts per strength level
# ============================================

try:
    from humanizer import humanize
    HAS_HUMANIZER = True
    print('Humanizer loaded successfully!')
except ImportError as e:
    HAS_HUMANIZER = False
    print(f'Could not load humanizer: {e}')
    print('Skipping humanizer test. Detector + patterns are still saved.')

results_by_strength = {}

if HAS_HUMANIZER:
    # Pick 200 real AI texts
    ai_test_indices = [i for i, l in enumerate(all_labels) if l == 1]
    np.random.seed(42)
    test_batch = np.random.choice(ai_test_indices, size=min(200, len(ai_test_indices)), replace=False)
    test_ai_texts = [all_texts[i] for i in test_batch]

    print(f'\nTesting humanizer on {len(test_ai_texts)} real AI texts...')

    strengths = ['light', 'medium', 'strong']

    for strength in strengths:
        print(f'\n{"="*50}')
        print(f'  Strength: {strength.upper()}')
        print('=' * 50)

        scores_before = []
        scores_after = []
        successes = 0

        for idx, text in enumerate(test_ai_texts[:50]):
            try:
                # Score original
                orig_features = profile_text(text)
                orig_vec = hstack([tfidf.transform([text]), pd.DataFrame([orig_features]).values])
                orig_score = lr_model.predict_proba(orig_vec)[0, 1]

                # Humanize
                humanized = humanize(text, strength=strength, stealth=True)

                # Score humanized
                hum_features = profile_text(humanized)
                hum_vec = hstack([tfidf.transform([humanized]), pd.DataFrame([hum_features]).values])
                hum_score = lr_model.predict_proba(hum_vec)[0, 1]

                scores_before.append(orig_score)
                scores_after.append(hum_score)
                if hum_score < 0.5:
                    successes += 1

                # Show first 3 examples
                if idx < 3:
                    status = 'PASS' if hum_score < 0.5 else 'FAIL'
                    print(f'  Sample {idx+1}: {orig_score:.0%} -> {hum_score:.0%}  [{status}]')
            except Exception as e:
                if idx < 3:
                    print(f'  Sample {idx+1}: Error - {str(e)[:50]}')

        results_by_strength[strength] = {
            'avg_before': float(np.mean(scores_before)) if scores_before else 0,
            'avg_after': float(np.mean(scores_after)) if scores_after else 0,
            'bypass_rate': float(successes / max(len(scores_after), 1)),
            'n_tested': len(scores_after),
        }

        r = results_by_strength[strength]
        print(f'\n  Average AI score: {r["avg_before"]:.0%} -> {r["avg_after"]:.0%}')
        print(f'  Bypass rate:      {r["bypass_rate"]:.0%} ({successes}/{r["n_tested"]})')

---
## STEP 10: Visualize Humanizer Results

Bar chart showing before/after AI detection scores for each strength level.

In [ ]:
# ============================================
# STEP 10: VISUALIZE RESULTS
# ============================================

if HAS_HUMANIZER and results_by_strength:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for idx, strength in enumerate(['light', 'medium', 'strong']):
        r = results_by_strength[strength]
        bars = axes[idx].bar(
            ['Before', 'After'],
            [r['avg_before']*100, r['avg_after']*100],
            color=['#e74c3c', '#2ecc71']
        )
        axes[idx].axhline(y=50, color='orange', linestyle='--', label='50% threshold')
        axes[idx].set_title(f'{strength.upper()}\nBypass: {r["bypass_rate"]:.0%}', fontsize=12)
        axes[idx].set_ylabel('AI Detection Score (%)')
        axes[idx].set_ylim(0, 100)
        axes[idx].legend()

        for bar, val in zip(bars, [r['avg_before']*100, r['avg_after']*100]):
            axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                          f'{val:.1f}%', ha='center', fontweight='bold')

    plt.suptitle('Humanizer vs Aggressive Detector', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/humanizer_vs_detector.png', dpi=150)
    plt.show()
    print('Plot saved!')
else:
    print('Humanizer was not loaded. No chart to show.')

---
## STEP 11: Run Built-in Trainer Loop (Optional)

This runs the existing `trainer.py` on your 6 local AI/human sample pairs with an aggressive target score of 80.

It iterates up to 10 times, auto-tuning rule parameters each round.

**Skip this if** the built-in trainer has missing dependencies.

In [ ]:
# ============================================
# STEP 11: RUN BUILT-IN TRAINER (OPTIONAL)
# Uses the 6 local sample pairs
# ============================================

train_dir = os.path.join(ENGINE_DIR, 'data', 'train')
best_checkpoint = None

if os.path.exists(train_dir):
    try:
        from trainer import Trainer, TrainerConfig

        config = TrainerConfig()
        config.max_iterations = 10
        config.threshold_score = 80   # aggressive target
        config.test_samples_dir = train_dir
        config.log_dir = '/content/training_logs'

        trainer = Trainer(config=config)
        best_checkpoint = trainer.run_training_loop()
        print('\nTrainer finished!')
    except Exception as e:
        print(f'Trainer error: {e}')
        print('This is OK. The detector and patterns from Steps 7-8 are still valid.')
else:
    print(f'No training pairs found at {train_dir}')
    print('Skipping built-in trainer.')

---
## STEP 12: Save 200 New Training Pairs

This extracts 200 matched AI/human text pairs from HC3 and Wikipedia datasets and saves them as individual `.txt` files.

You can copy these to `humanizer-engine/data/train/` later for more training data.

In [ ]:
# ============================================
# STEP 12: SAVE EXTENDED TRAINING PAIRS
# Creates 200 new AI/human text pairs
# ============================================

new_pairs_dir = '/content/extended_training_pairs'
os.makedirs(new_pairs_dir, exist_ok=True)

pair_count = 0

# From HC3 (matched Q&A pairs)
try:
    for _, row in hc3_df.iterrows():
        if pair_count >= 100:
            break
        human_ans = row.get('human_answers', [])
        ai_ans = row.get('chatgpt_answers', [])

        if (isinstance(human_ans, list) and human_ans and
            isinstance(ai_ans, list) and ai_ans):
            h = human_ans[0] if isinstance(human_ans[0], str) else ''
            a = ai_ans[0] if isinstance(ai_ans[0], str) else ''

            if len(h) > 100 and len(a) > 100:
                pair_count += 1
                with open(os.path.join(new_pairs_dir, f'pair_{pair_count:03d}_human.txt'), 'w', encoding='utf-8') as f:
                    f.write(h)
                with open(os.path.join(new_pairs_dir, f'pair_{pair_count:03d}_ai.txt'), 'w', encoding='utf-8') as f:
                    f.write(a)
except Exception as e:
    print(f'HC3 extraction note: {e}')

# From Wikipedia vs GPT
try:
    for _, row in wiki_sample.iterrows():
        if pair_count >= 200:
            break
        h = str(row.get('wiki_intro', ''))
        a = str(row.get('generated_intro', ''))
        if len(h) > 100 and len(a) > 100:
            pair_count += 1
            with open(os.path.join(new_pairs_dir, f'pair_{pair_count:03d}_human.txt'), 'w', encoding='utf-8') as f:
                f.write(h)
            with open(os.path.join(new_pairs_dir, f'pair_{pair_count:03d}_ai.txt'), 'w', encoding='utf-8') as f:
                f.write(a)
except Exception as e:
    print(f'Wiki extraction note: {e}')

print(f'Saved {pair_count} AI/human text pairs to {new_pairs_dir}')
print(f'\nYou can copy these to humanizer-engine/data/train/ for future training.')

---
## STEP 13: Test on YOUR Own Text

**Edit the text in the cell below** — replace the sample text with any AI-generated text you want to test.

It will:
1. Score the original text with the aggressive detector
2. Humanize it at medium and strong strength
3. Score the humanized versions
4. Show you if it passes (score < 50%)

In [ ]:
# ============================================
# STEP 13: TEST YOUR OWN TEXT
# >>> EDIT THE TEXT BELOW <<<
# ============================================

# === PASTE YOUR AI TEXT HERE (between the triple quotes) ===
my_text = """
The intersection of artificial intelligence and natural language processing 
has become a focal point of contemporary research. Furthermore, the implications 
of these technological advances extend far beyond the technical realm. Moreover, 
society faces unprecedented challenges in distinguishing authentic human discourse 
from machine-generated content. In conclusion, the urgency to develop robust 
detection mechanisms cannot be overstated. It is important to note that the 
current landscape presents both opportunities and risks.
""".strip()
# === END OF YOUR TEXT ===


# Score original
orig_feat = profile_text(my_text)
orig_vec = hstack([tfidf.transform([my_text]), pd.DataFrame([orig_feat]).values])
orig_ai_score = lr_model.predict_proba(orig_vec)[0, 1]

print('=' * 60)
print('  ORIGINAL TEXT')
print('=' * 60)
print(my_text[:500])
print(f'\n  AI Detection Score: {orig_ai_score:.1%}')
print(f'  Verdict: {"DETECTED AS AI" if orig_ai_score > 0.5 else "Looks human"}')

# Test humanizer if available
if HAS_HUMANIZER:
    for strength in ['medium', 'strong']:
        print(f'\n{"="*60}')
        print(f'  HUMANIZED ({strength.upper()})')
        print('=' * 60)

        humanized = humanize(my_text, strength=strength, stealth=True)
        print(humanized[:500])

        hum_feat = profile_text(humanized)
        hum_vec = hstack([tfidf.transform([humanized]), pd.DataFrame([hum_feat]).values])
        hum_ai_score = lr_model.predict_proba(hum_vec)[0, 1]

        drop = orig_ai_score - hum_ai_score
        status = 'PASS - Bypassed detector!' if hum_ai_score < 0.5 else 'FAIL - Still detected'
        print(f'\n  AI Score: {hum_ai_score:.1%} (dropped {drop:.1%} from original)')
        print(f'  Verdict: {status}')
else:
    print('\nHumanizer not loaded - showing detector score only.')

---
## STEP 14: Download Everything as ZIP

This packages all outputs into a single ZIP file and downloads it:

| File | What It Contains |
|------|------------------|
| `aggressive_detector.pkl` | Trained AI detector model |
| `ai_pattern_report.json` | Top 30 AI + 30 human indicators |
| `ai_vs_human_features.png` | Feature distribution charts |
| `humanizer_vs_detector.png` | Before/after humanizer results |
| `humanizer_test_results.json` | Bypass rates per strength level |
| `optimized_rules.json` | Tuned rule parameters (if trainer ran) |

**Run both cells below** — first one packages, second one downloads.

In [ ]:
# ============================================
# STEP 14a: PACKAGE ALL OUTPUTS
# ============================================

import shutil

export_dir = '/content/humara_aggressive_export'
os.makedirs(export_dir, exist_ok=True)

# Copy all outputs
files_to_copy = [
    '/content/models/aggressive_detector.pkl',
    '/content/ai_pattern_report.json',
    '/content/ai_vs_human_features.png',
    '/content/humanizer_vs_detector.png',
]

for fpath in files_to_copy:
    if os.path.exists(fpath):
        shutil.copy(fpath, export_dir)

if os.path.exists('/content/training_logs/training_report.json'):
    shutil.copy('/content/training_logs/training_report.json', export_dir)

# Save optimized rules if available
try:
    if best_checkpoint and 'rules' in best_checkpoint:
        with open(os.path.join(export_dir, 'optimized_rules.json'), 'w') as f:
            json.dump(best_checkpoint['rules'], f, indent=2)
except:
    pass

# Save humanizer test results
try:
    if results_by_strength:
        with open(os.path.join(export_dir, 'humanizer_test_results.json'), 'w') as f:
            json.dump(results_by_strength, f, indent=2)
except:
    pass

# Create ZIP
shutil.make_archive('/content/humara_aggressive_export', 'zip', export_dir)

print('Export ready! Files in ZIP:')
print('-' * 40)
for f in sorted(os.listdir(export_dir)):
    size = os.path.getsize(os.path.join(export_dir, f))
    print(f'  {f} ({size/1024:.1f} KB)')

In [ ]:
# ============================================
# STEP 14b: DOWNLOAD THE ZIP
# This will trigger a browser download
# ============================================

from google.colab import files
files.download('/content/humara_aggressive_export.zip')

print()
print('=' * 50)
print('  DONE!')
print('=' * 50)
print()
print('Next steps:')
print('  1. Unzip the downloaded file')
print('  2. Copy ai_pattern_report.json to your project')
print('  3. Use the patterns to improve your humanizer rules')
print('  4. Copy aggressive_detector.pkl to test against')